# Facet Evaluator - GPU Scoring Pipeline (Google Colab)

This notebook runs the **Facet Evaluator** scoring pipeline on Google Colab across **20 Tricky Anti-Hallucination Scenarios** using **Qwen2.5-7B-Instruct / 14B-Instruct** on a GPU instance and exports genuine LLM evaluation reports to JSON.

## Step 0: Setup Project Directory in Colab

If you uploaded or cloned your repository in Colab, navigate to your repository folder below.

In [ ]:
# Run this if cloning directly in Colab
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/ai_ml_project
# %cd /content/ai_ml_project

## Step 1: GPU Verification & Dependency Installation

In [ ]:
!nvidia-smi
!pip install -q pandas pydantic sentence-transformers transformers accelerate bitsandbytes torch

## Step 2: Configure Repository Root & Python sys.path

In [ ]:
import os
import sys
from pathlib import Path

root_dir = None
candidates = [
    Path(os.getcwd()).resolve(),
    Path(os.getcwd()).resolve().parent,
    Path("/content/ai_ml_project"),
    Path("/content/facet-evaluator"),
    Path("/content")
]
for c in candidates:
    if (c / "src").exists() and (c / "src").is_dir():
        root_dir = c
        break

if root_dir is None:
    raise FileNotFoundError("The 'src' folder was not found on the Colab server! Please upload 'src/' and 'data/' folders or run git clone first.")

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

print(f"Repository Root set to: {root_dir}")
print(f"'src' exists: {(root_dir / 'src').exists()}")

## Step 3: Initialize Open-Weight GPU Model (Qwen2.5-7B-Instruct)

In [ ]:
from src.scoring.inference_backend import HuggingFaceInferenceBackend

gpu_backend = HuggingFaceInferenceBackend(
    model_id="Qwen/Qwen2.5-7B-Instruct",
    load_in_4bit=True
)
print("Genuine GPU LLM Model Loaded Successfully!")

## Step 4: Run 20 Tricky Anti-Hallucination Scenarios & Export Output to JSON

In [ ]:
import json
from src.evaluator_pipeline import FacetEvaluatorPipeline

TRICKY_BENCHMARK_SCENARIOS = [
    {"id": "SCENARIO_001", "description": "Medical Symptom Query (Diabetes/Blood test hallucination trap)", "conversation_text": "I have been having frequent headaches and feeling thirsty all the time."},
    {"id": "SCENARIO_002", "description": "External Biographical Car Ownership Trap (Traffic statement)", "conversation_text": "I spent two hours stuck in terrible bumper-to-bumper traffic on Highway 101 today."},
    {"id": "SCENARIO_003", "description": "Emotional Mood vs Clinical Construct (Depression scale trap)", "conversation_text": "I am feeling really blue and down in the dumps today because it's raining."},
    {"id": "SCENARIO_004", "description": "Metaphorical Job Risk & Decisiveness", "conversation_text": "I took a huge leap of faith and submitted my resignation letter without having another job lined up!"},
    {"id": "SCENARIO_005", "description": "Passive-Aggressive Interpersonal Posture", "conversation_text": "Fine, do whatever you want. It's not like my opinion ever mattered to anyone here anyway."},
    {"id": "SCENARIO_006", "description": "Spiritual Retreat vs Esoteric I Ching Hexagram Trap", "conversation_text": "I spent the weekend quietly meditating at a peaceful retreat in the mountains."},
    {"id": "SCENARIO_007", "description": "Mental Math Claim vs Standardized Cognitive Test Trap", "conversation_text": "I can calculate 15% tip in my head in two seconds flat!"},
    {"id": "SCENARIO_008", "description": "Digital Nomad Lifestyle vs External Passport Log Trap", "conversation_text": "I've been working remotely from coffee shops in Bali and Bangkok for the last six months."},
    {"id": "SCENARIO_009", "description": "Financial Risk & Cryptocurrency Impulsivity", "conversation_text": "I put my entire life savings into a high-volatility meme cryptocurrency yesterday."},
    {"id": "SCENARIO_010", "description": "Extreme Brevity & Laconic Directness", "conversation_text": "No."},
    {"id": "SCENARIO_011", "description": "Overprotectiveness vs Home Security Hardware Log Trap", "conversation_text": "I installed five security cameras, three locks, and I track my daughter's GPS location 24/7."},
    {"id": "SCENARIO_012", "description": "Sarcasm, Acidity & Civility", "conversation_text": "Oh sure, I'd just LOVE to stay past midnight fixing your typos for free again!"},
    {"id": "SCENARIO_013", "description": "Religious Self-Reflection vs Memorization Count Trap", "conversation_text": "I read scripture every single morning before starting my day."},
    {"id": "SCENARIO_014", "description": "Dietary Habit Claim vs Biological Macronutrient Ratio Trap", "conversation_text": "I haven't eaten processed food or sugar in over three years."},
    {"id": "SCENARIO_015", "description": "High Hesitation & Cognitive Deliberation", "conversation_text": "Um... well... I guess maybe we could... wait, let me think... maybe option B?"},
    {"id": "SCENARIO_016", "description": "Chivalry & Warmheartedness", "conversation_text": "I held the elevator door for an elderly stranger and carried their heavy groceries up the stairs."},
    {"id": "SCENARIO_017", "description": "Direct Confrontation vs Non-Conformity", "conversation_text": "I directly confronted my manager about the unfair budget cuts during our team meeting."},
    {"id": "SCENARIO_018", "description": "Emotional Burnout & Exhaustion", "conversation_text": "I am completely exhausted, emotionally drained, and I can barely force myself to open my laptop."},
    {"id": "SCENARIO_019", "description": "Creative DIY Activity vs Pet Tracking Sensor Trap", "conversation_text": "I built a custom agility obstacle course in my backyard for my Golden Retriever."},
    {"id": "SCENARIO_020", "description": "Hypothetical Counterfactual Threat Posture", "conversation_text": "If I were ever in a robbery, I would probably freeze and give them all my money."}
]

pipeline = FacetEvaluatorPipeline(backend=gpu_backend, top_k=10, batch_size=5)
pipeline.initialize()

genuine_json_output = {}

for i, sc in enumerate(TRICKY_BENCHMARK_SCENARIOS, 1):
    sid = sc["id"]
    text = sc["conversation_text"]
    print(f"[{i}/20] Evaluating genuine GPU output for {sid}: '{text}'...")
    res = pipeline.evaluate_conversation(text)
    
    genuine_json_output[sid] = {
        "scenario_description": sc["description"],
        "conversation_text": text,
        "total_candidates_retrieved": res.total_candidates_retrieved,
        "evaluated_results": [item.model_dump() for item in res.evaluated_results],
        "abstained_results": [item.model_dump() for item in res.abstained_results]
    }

out_file = root_dir / "outputs" / "evaluation_results.json"
out_file.parent.mkdir(parents=True, exist_ok=True)
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(genuine_json_output, f, indent=2, ensure_ascii=False)

print(f"\nSUCCESS! Saved Genuine GPU LLM Report for 20 Tricky Scenarios to: {out_file}")